In [ ]:
import torch
import numpy as np
from load_dataset import LoadDataset
from model import MLP
from torch.utils.data import DataLoader
from training import Trainer
from metrics import Evaluator
from gradients import GradMonitor 
import matplotlib.pyplot as plt  

In [2]:
iris_path = '/home/jessica/Documents/pesquisa/knowledge_distillation/dataset/Iris.csv'
iris = LoadDataset(iris_path)
iris.load()
train_data, test_data = iris.split_dataset()

In [3]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [4]:
config_iris_MLP = {
    'input_dim': 4,
    'in_channels': 1,   
    'output_dim': 3,     # RGB
    'fc': [5],
    'criterion':  torch.nn.CrossEntropyLoss(),
    'optimizer':  torch.optim.Adam,
    'lr': 0.001
}

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
student = MLP(config_iris_MLP).to(device)
teacher = MLP(config_iris_MLP).to(device)
trainer = Trainer(teacher, config_iris_MLP)
epochs = 1000
ptrain_old = Evaluator(teacher).accuracy(train_loader)
patience = 0
for _ in range(epochs):
    trainer.train_epoch(train_loader)
    ptrain = Evaluator(teacher).accuracy(train_loader)
    if abs(ptrain_old - ptrain) < 1e-3:  # Convergence check
        patience += 1
    else:
        patience = 0
    ptrain_old = ptrain
    if patience >= 60:  # Early stopping after 5 epochs without improvement
        print(f"Early stopping at epoch {_+1}")
        break
metrics = Evaluator(teacher)
print(metrics.accuracy(train_loader), metrics.accuracy(test_loader))
grad_teacher = GradMonitor(teacher, config_iris_MLP)
info_teacher = grad_teacher.get_gradients(train_loader)


Early stopping at epoch 215
0.9904761904761905 0.9555555555555556


In [6]:
def get_student_model(T, c, teacher, train_loader):
    student = MLP(config_iris_MLP).to(device)
    t_student = Trainer(student, config_iris_MLP)
    sepochs = 1000
    ptrain = Evaluator(teacher).accuracy(train_loader)
    for k in range(sepochs):
        t_student.train_student(teacher, train_loader, T=1, c=0.5)
        strain = Evaluator(student).accuracy(train_loader)
        if abs(strain - ptrain) < 0.001:  # Convergence check
            print(k)
            break
    print(ptrain, Evaluator(teacher).accuracy(test_loader))  
    print(strain, Evaluator(student).accuracy(test_loader))
    return student

st_models = {}
for T in [1, 5, 10]:
    for c in [0, 0.5, 0.9, 1]:
        print(f"Training student with T={T} and c={c}")
        student = get_student_model(T, c, teacher, train_loader)
        st_models[f'T={T}, c={c}'] = student

Training student with T=1 and c=0


664
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9111111111111111
Training student with T=1 and c=0.5
387
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333
Training student with T=1 and c=0.9
496
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333
Training student with T=1 and c=1
483
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333
Training student with T=5 and c=0
463
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9555555555555556
Training student with T=5 and c=0.5
643
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9111111111111111
Training student with T=5 and c=0.9
562
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333
Training student with T=5 and c=1
407
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333
Training student with T=10 and c=0
560
0.9904761904761905 0.9555555555555556
0.9904761904761905 0.9333333333333333

##### Será que as informações dentro as redes sao iguais? -> vou dar uma olhada nos gradientes
---

In [7]:
teacher_grad_all = {}
student_grad_all = {}
output_student_all = {}
output_teacher_all = {}
pred_student_all = {}
pred_teacher_all = {}
for params in st_models:
    grad_student = GradMonitor(st_models[params], config_iris_MLP)
    info_student = grad_student.get_gradients(train_loader) 
    teacher_grad = []
    student_grad = []
    output_teacher = []
    output_student = []
    pred_teacher = []
    pred_student = []
    for i in range(len(info_student)):
        teacher_grad.append(info_teacher[i]['grads']['layers.fc1.weight'])
        student_grad.append(info_student[i]['grads']['layers.fc1.weight'])
        output_student.append(info_student[i]['grads']['layers.fc2.weight'])
        output_teacher.append(info_teacher[i]['grads']['layers.fc2.weight'])
        pred_student.append(info_student[i]['pred_labels'])
        pred_teacher.append(info_teacher[i]['pred_labels'])
    teacher_grad = torch.cat(teacher_grad, dim=0)
    student_grad = torch.cat(student_grad, dim=0)
    output_teacher = torch.cat(output_teacher, dim=0)
    output_student = torch.cat(output_student, dim=0)
    pred_teacher = torch.cat(pred_teacher, dim=0)
    pred_student = torch.cat(pred_student, dim=0)  
    torder = np.argsort(pred_teacher.cpu().numpy())
    sorder = np.argsort(pred_student.cpu().numpy())
    teacher_grad_sorted = teacher_grad[torder]
    student_grad_sorted = student_grad[sorder]
    output_student_sorted = output_student[sorder]
    output_teacher_sorted = output_teacher[torder] 
    pred_student_all[params] = pred_student
    pred_teacher_all[params] = pred_teacher
    teacher_grad_all[params] = teacher_grad_sorted
    student_grad_all[params] = student_grad_sorted
    output_student_all[params] = output_student_sorted
    output_teacher_all[params] = output_teacher_sorted  

In [8]:
from pruning_utils import compute_similarity_matrix

In [9]:
all_teacher_similarity = {}
all_student_similarity = {}
all_output_teacher_similarity = {}
all_output_student_similarity = {}
for params in student_grad_all:
    teacher_similarity = compute_similarity_matrix(teacher_grad_all[params])
    student_similarity = compute_similarity_matrix(student_grad_all[params])
    output_teacher_similarity = compute_similarity_matrix(output_teacher_all[params])
    output_student_similarity = compute_similarity_matrix(output_student_all[params])
    all_teacher_similarity[params] = teacher_similarity
    all_student_similarity[params] = student_similarity
    all_output_teacher_similarity[params] = output_teacher_similarity
    all_output_student_similarity[params] = output_student_similarity